# SrRuO3 raw HDF5: download, inspect, and preserve

Runs against the real pinned G5 recording in the SrRuO3 Data project (3.70 GB download). Reads a small frame window; this validates data access and provenance, not the full plume-metrics re-analysis. Saves a table, NumPy frame, rendered plot, original SVG, executed notebook, and linked run summary.

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "dataerai_preservation.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from dataerai_preservation import start_analysis
dataerai = start_analysis(REPO_ROOT / 'notebooks/00_Dataerai_Raw_Data_Quickstart.ipynb', input_paths=['Plumes/G5.h5'], include_legacy=False)


In [ ]:
%%dataerai
import h5py
import numpy as np
import pandas as pd
from IPython.display import display
from pathlib import Path

with h5py.File('../data/Plumes/G5.h5', 'r') as recording:
    inventory = []
    def inspect_dataset(name, value):
        if isinstance(value, h5py.Dataset):
            inventory.append({'dataset': name, 'shape': str(value.shape), 'dtype': str(value.dtype)})
    recording.visititems(inspect_dataset)
    dataset = recording['PLD_Plumes/1-SrRuO3']
    frame = np.asarray(dataset[0, 0, :128, :128])
dataset_inventory = pd.DataFrame(inventory)
display(dataset_inventory)
print('Real raw-frame window:', frame.shape, frame.dtype)


In [ ]:
%%dataerai
frame_statistics = pd.DataFrame([{'sample': 'G5', 'frame': 0, 'time_index': 0,
    'pixels': frame.size, 'minimum': int(frame.min()), 'maximum': int(frame.max()),
    'mean': float(frame.mean()), 'standard_deviation': float(frame.std())}])
np.save('../data/G5-frame-window.npy', frame)
frame_statistics.to_csv('../data/G5-frame-statistics.csv', index=False)
dataset_inventory.to_json('../data/G5-dataset-inventory.json', orient='records', indent=2)
display(frame_statistics)


In [ ]:
%%dataerai
%matplotlib inline
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(frame, cmap='viridis')
ax.set(title='G5: first raw frame, 128×128 window', xlabel='x (pixel)', ylabel='y (pixel)')
fig.colorbar(image, ax=ax, label='Recorded intensity')
fig.tight_layout()
fig.savefig('../figures/G5-raw-frame-window.svg')
plt.show()


In [ ]:
dataerai.finish()
